# Functional Programming in Python

---

> **The story:** We start with a simple problem — computing a factorial — and tell it four different ways. Along the way, we pick up the core tools of functional programming: **lambda**, **iterators**, **generators**, **filter**, **map**, **reduce**, **closures**, and **currying**.

---

## Chapter 1 — Lambda (Anonymous Functions)

In Python, a normal function has a name. A **lambda** is a nameless, one-line function. Think of it as a sticky note — quick, disposable, useful on the spot.

```
Syntax:  lambda arguments : expression
```

In [ ]:
# Regular function
def square(x):
    return x * x

# Equivalent lambda
square_l = lambda x: x * x

print(square(5))    # 25
print(square_l(5))  # 25

25
25


In [34]:
# Lambda with two arguments
multiply = lambda x, y: x * y if x < y else x / y
print(multiply(6, 3))  # 12

# Lambda used inline (no variable needed)

print((lambda x: x + 10)(5))  # 15

2.0
15


> **Key idea:** Lambdas are most useful when passed directly into other functions like `map`, `filter`, and `reduce` — as you'll see shortly.

---
## Chapter 2 — Iterators

An **iterator** is an object that produces values one at a time when you call `next()` on it. Python lists, ranges, and strings are all *iterable* — you can get an iterator from them.

Think of an iterator as a bookmark in a book: it remembers where you are and gives you the next page on demand.

In [38]:
numbers = [10, 20, 30, 40 ,50]

# for i in numbers:
#     print(i)

it = iter(numbers)       # Create an iterator from the list
it2 = iter(numbers)
print(next(it))          # 10
print(next(it))          # 20
print("hello  world")
print(next(it2))
print(next(it))          # 30
# print(next(it))        # Would raise StopIteration

10
20
hello  world
10
30


> **Key idea:** Iterators are *lazy* — they don't compute all values upfront. They produce one value at a time. Generators take this idea further.

---
## Chapter 3 — Generators

A **generator** is a special function that uses `yield` instead of `return`. Each time you call `next()`, it runs until the next `yield`, pauses, and resumes from there next time.

This makes generators **memory-efficient** — they never build the full sequence in memory.

In [47]:
# A generator that yields numbers 1 to n
def count_up(n):
    i = 1
    while i <= n:
        yield i        # Pause here, give back i
        i += 1

gen = count_up(5)
gen2 = count_up(7)
print(next(gen))  # 1
# print(next(gen2))  # 1
print(next(gen))  # 2
print(next(gen))  # 2
print(next(gen))  # 2
print(next(gen))  # 2
print(next(gen2))  # 2
# print(list(gen))  # [3, 4, 5]  — rest of the values

1
2
3
4
5
1


In [75]:
# Fibonacci generator (like the slide you saw)
def fibonacci():
    a, b = 0, 1
    while True:         # Infinite sequence!
        yield a
        a, b = b, a + b

fib = fibonacci()
# next(fib)
print([next(fib) for _ in range(8)])  # First 8 Fibonacci numbers

121393

> **Key idea:** A generator can represent an *infinite* sequence without crashing your memory — because it only computes the next value when asked. This is lazy evaluation in Python.

---
## Chapter 4 — filter, map, reduce

These three functions are the workhorses of functional programming. Instead of writing loops, you *describe* what transformation you want.

Let's use a single list of numbers throughout to see how they connect:

```python
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
```

### 4a — filter()

`filter(function, iterable)` — keeps only elements where the function returns `True`.

> Think of it as a sieve: it lets some things through and blocks others.

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Keep only even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(evens)  # [2, 4, 6, 8, 10]

# filter() returns an iterator — wrap in list() to see all values

[2, 4, 6, 8, 10]


### 4b — map()

`map(function, iterable)` — applies a function to *every* element and returns the transformed sequence.

> Think of it as an assembly line: every item gets the same operation applied to it.

In [ ]:
# Square every number
squared = list(map(lambda x: x ** 2, numbers))
print(squared)  # [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

# Chain filter + map: square only the even numbers
result = list(map(lambda x: x ** 2, filter(lambda x: x % 2 == 0, numbers)))
print(result)  # [4, 16, 36, 64, 100]

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
[4, 16, 36, 64, 100]


### 4c — reduce()

`reduce(function, iterable)` — collapses a sequence into a *single value* by repeatedly applying a function to accumulate results.

> Think of it as a snowball rolling downhill — it picks up more as it goes, ending as one big value.

In [ ]:
from functools import reduce

# Sum all numbers:  ((((1+2)+3)+4)+...+10)
total = reduce(lambda acc, x: acc + x, numbers)
print(total)  # 55

# Find the maximum value
maximum = reduce(lambda a, b: a if a > b else b, numbers)
print(maximum)  # 10

55
10


---
## Chapter 5 — Closure

A **closure** is a function that *remembers* variables from the scope where it was created, even after that scope has finished.

> Think of it like a backpack: the inner function carries the outer variable with it wherever it goes.

In [76]:
def make_multiplier(factor):
    # 'factor' lives in the outer scope
    def multiply(x):
        return x * factor   # inner function 'captures' factor
    return multiply

# print(make_multiplier(10))

double = make_multiplier(20)
triple = make_multiplier(3)

print(double(5))   # 10  — factor=2 is remembered
print(triple(5))   # 15  — factor=3 is remembered

# Even though make_multiplier() has finished, 'factor' lives on inside the closure

100
15


In [77]:
# Closure for a simple counter (data encapsulation without a class)
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    def reset():
        nonlocal count
        count = 0
    return increment, reset

inc, rst = make_counter()
print(inc())  # 1
print(inc())  # 2
print(inc())  # 3
rst()
print(inc())  # 1  — reset worked

1
2
3
1


> **`nonlocal`** tells Python: "this variable belongs to the enclosing function's scope, not a new local one". Without it, `count += 1` would raise an `UnboundLocalError`.

---
## Chapter 6 — Currying

**Currying** transforms a function that takes multiple arguments into a chain of functions, each taking one argument.

> Instead of `add(3, 5)`, currying gives you `add(3)(5)` — you feed arguments one at a time.

**Partial application** is related but different: you fix *some* arguments now and supply the rest later (not necessarily one at a time).

In [81]:
# Normal two-argument function
# def add(x, y):
#     return x + y

# Curried version — returns a function waiting for the second argument
def add_curried(x):
    def inner(y):
        return x + y
    return inner

# add5 = add_curried(5)    # Fix x=5
# print(add5(3))           # 8
# print(add5(10))          # 15
print(add_curried(2)(7)) # 9  — call directly

# print(add(2,7))

9


In [82]:
# Using functools.partial for partial application
from functools import partial

def power(base, exp):
    return base ** exp

square  = partial(power, exp=2)   # Fix exp=2
# cube    = partial(power, exp=3)   # Fix exp=3

print(square(4))   # 16
# print(cube(3))     # 27

# Currying: one arg at a time
# Partial application: fix any subset of args

16


---
## Chapter 7 — The Factorial Story

Now let's tie everything together with one classic problem: **factorial**.

We'll solve it four ways, moving from imperative to progressively more functional.

```
4! = 4 × 3 × 2 × 1 = 24
```

### Way 1 — Imperative (Loop)

The classic approach: maintain a `result` variable and mutate it inside a loop. Step-by-step, stateful.

In [ ]:
def factorial_loop(n):
    if n < 0:
        return "Not defined for negative numbers"
    result = 1
    for i in range(1, n + 1):
        result *= i          # State changes each iteration
    return result

print(factorial_loop(5))  # 120
print(factorial_loop(4))  # 24

120
24


> **Imperative style:** *how* to compute — explicit steps, mutable state, loop.

### Way 2 — Functional (Recursion)

No loop. No mutation. The function calls itself with a smaller problem each time.

```
factorial(4)
= 4 × factorial(3)
= 4 × (3 × factorial(2))
= 4 × (3 × (2 × factorial(1)))
= 4 × (3 × (2 × (1 × factorial(0))))
= 4 × (3 × (2 × (1 × 1)))
= 24
```

In [ ]:
def factorial_recursive(n):
    return 1 if n <= 0 else n * factorial_recursive(n - 1)

print(factorial_recursive(5))  # 120
print(factorial_recursive(4))  # 24

120
24


> **Functional style:** *what* to compute — no mutation, expressive, but each call adds a frame to the call stack.

### Way 3 — Functional (Tail Recursion)

The recursive call is the *very last* thing the function does — no pending multiplication after it returns. An accumulator carries the running result.

```
factorial(4)
= f(4, 1)
= f(3, 4)
= f(2, 12)
= f(1, 24)
= 24
```

In [ ]:
def factorial_tail(n):
    def f(n, acc):               # acc = accumulator
        return acc if n <= 0 else f(n - 1, n * acc)
    return f(n, 1)

print(factorial_tail(5))  # 120
print(factorial_tail(4))  # 24

120
24


> **Why tail recursion?** In languages that optimise tail calls (like Haskell or Scala), no new stack frames are added — it runs as efficiently as a loop. Python doesn't optimise tail calls, but the *pattern* is important conceptually.

### Way 4 — Functional (reduce)

Use `reduce` to collapse the range `[1, 2, 3, 4, 5]` into a single value by multiplying.

```
reduce(×, [1,2,3,4,5])  →  ((((1×2)×3)×4)×5)  =  120
```

In [ ]:
from functools import reduce

def factorial_reduce(n):
    return reduce(lambda x, y: x * y, range(1, n + 1)) if n > 0 else 1

print(factorial_reduce(5))  # 120
print(factorial_reduce(4))  # 24

120
24


> **reduce** takes a function and a collection, and returns a single value created by combining the items — exactly what we need here.

### Side-by-side Comparison

In [ ]:
n = 6
print(f"Loop:          {factorial_loop(n)}")
print(f"Recursion:     {factorial_recursive(n)}")
print(f"Tail recursion:{factorial_tail(n)}")
print(f"Reduce:        {factorial_reduce(n)}")
# All should print 720

Loop:          720
Recursion:     720
Tail recursion:720
Reduce:        720


---
## Chapter 8 — Putting It All Together

One final pipeline that uses **filter + map + reduce + lambda** together — the full functional toolkit in one flow.

In [ ]:
from functools import reduce

scores = [45, 67, 89, 23, 91, 56, 78, 34, 82, 95]

# Step 1: filter — keep passing scores (>= 50)
passing = list(filter(lambda s: s >= 50, scores))
print("Passing:", passing)

# Step 2: map — add 5 bonus points
boosted = list(map(lambda s: s + 5, passing))
print("Boosted:", boosted)

# Step 3: reduce — compute total, then average
total = reduce(lambda acc, s: acc + s, boosted)
average = total / len(boosted)
print(f"Average boosted score: {average:.2f}")

Passing: [67, 89, 91, 56, 78, 82, 95]
Boosted: [72, 94, 96, 61, 83, 87, 100]
Average boosted score: 84.71


Now add a **closure** to create a reusable bonus-adder:

In [ ]:
# Closure: make_bonus_adder captures 'bonus'
def make_bonus_adder(bonus):
    return lambda score: score + bonus

add_10 = make_bonus_adder(10)
add_5  = make_bonus_adder(5)

boosted_10 = list(map(add_10, passing))
print("With +10 bonus:", boosted_10)

# Curried bonus adder
def bonus_adder(bonus):
    def apply(score):
        return score + bonus
    return apply

print("Curried +7:", list(map(bonus_adder(7), passing)))

With +10 bonus: [77, 99, 101, 66, 88, 92, 105]
Curried +7: [74, 96, 98, 63, 85, 89, 102]


---
## Summary Table

| Concept | What it does | Key syntax |
|---|---|---|
| **Lambda** | Nameless one-line function | `lambda x: x * 2` |
| **Iterator** | Produces values one at a time | `iter()`, `next()` |
| **Generator** | Lazy sequence with `yield` | `def f(): yield x` |
| **filter()** | Keeps elements passing a test | `filter(lambda x: x>0, lst)` |
| **map()** | Transforms every element | `map(lambda x: x*2, lst)` |
| **reduce()** | Collapses sequence to one value | `reduce(lambda a,b: a+b, lst)` |
| **Closure** | Function that remembers outer variables | `def outer(): def inner(): ...` |
| **Currying** | Chain of single-argument functions | `add(3)(5)` |

---

> **One thought to carry forward:** In functional programming, you *describe* transformations rather than *command* steps. Combine small pure functions — and your code becomes easier to read, test, and reason about.